# UFO Sightings: Altair + Vega-Lite visualizations

This notebook builds two visualizations of the **UFO sightings** dataset (NUFORC reports, geocoded and time-standardized) for the IS445 Jekyll homework.

1. **When and what?** An interactive timeline of sightings per year linked to a bar chart of reported UFO shapes (brush the timeline, click the bars).
2. **Where?** A map of sightings across the continental US, with a dropdown to filter by shape.

Both charts are saved as Vega-Lite JSON into `assets/json/` so the Jekyll site can embed them.

In [1]:
import pandas as pd
import altair as alt

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## 1. Read the data from a URL

The CSV has no header row, so the column names are supplied here.

In [2]:
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

cols = ["datetime", "city", "state", "country", "shape",
        "duration_seconds", "duration_text", "comments",
        "date_posted", "latitude", "longitude"]

ufo = pd.read_csv(url, header=None, names=cols, low_memory=False)
print(ufo.shape)
ufo.head()

(80332, 11)


,datetime,city,state,country,shape,duration_seconds,duration_text,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


## 2. Clean and transform

- **Timestamps:** a few rows use `24:00` as the time, which pandas can't parse. I swap it for `00:00`, then convert with `pd.to_datetime` and pull out `year`.
- **Coordinates:** `latitude`/`longitude` are cast to numbers (a few bad values become `NaN` and get dropped).
- **Shapes:** there are ~30 shape labels, many very rare. I merge near-duplicates (e.g. `changed`/`changing`, `flare`/`flash`), fill missing values with `unknown`, and keep the **9 most common shapes**, lumping everything else into `other`. This keeps the color legend readable.
- **Scope:** I keep US reports only (country `us`, or blank country with a US state code) and drop 2014 because the data stops partway through that year.

In [3]:
ufo["datetime"] = pd.to_datetime(ufo["datetime"].str.replace("24:00", "00:00"),
                                 format="%m/%d/%Y %H:%M", errors="coerce")
ufo["latitude"] = pd.to_numeric(ufo["latitude"], errors="coerce")
ufo["longitude"] = pd.to_numeric(ufo["longitude"], errors="coerce")
ufo = ufo.dropna(subset=["datetime", "latitude", "longitude"])
ufo["year"] = ufo["datetime"].dt.year

# tidy the shape labels
ufo["shape"] = (ufo["shape"].fillna("unknown").str.strip().str.lower()
                .replace({"changed": "changing", "flare": "flash",
                          "delta": "triangle", "pyramid": "triangle",
                          "round": "circle", "crescent": "other"}))

top_shapes = [s for s in ufo["shape"].value_counts().index if s != "other"][:9]
ufo["shape_group"] = ufo["shape"].where(ufo["shape"].isin(top_shapes), "other")

us_states = ["al","ak","az","ar","ca","co","ct","de","dc","fl","ga","hi","id","il","in","ia","ks","ky",
             "la","me","md","ma","mi","mn","ms","mo","mt","ne","nv","nh","nj","nm","ny","nc","nd","oh",
             "ok","or","pa","ri","sc","sd","tn","tx","ut","vt","va","wa","wv","wi","wy"]
is_us = (ufo["country"] == "us") | (ufo["country"].isna() & ufo["state"].isin(us_states))
us = ufo[is_us & ufo["state"].isin(us_states) & ufo["year"].between(1950, 2013)].copy()

print(len(us), "US sightings between 1950 and 2013")
print(us["shape_group"].value_counts())

68694 US sightings between 1950 and 2013
shape_group
other        15131
light        14175
triangle      6895
unknown       6514
circle        6337
fireball      5369
sphere        4556
disk          4439
oval          3199
formation     2079
Name: count, dtype: int64


## 3. Plot 1: Sightings over time, by shape (interactive)

Plotting 65k individual rows would be slow and make a huge JSON file, so I aggregate first: **one row per (year, shape)** with a count. Vega-Lite then re-sums those counts after each filter.

**Interactivity**
- Drag across the top timeline to **brush a range of years**. The bar chart below updates to show the shape mix for just those years.
- **Click a bar** (shift-click for more than one) to filter the timeline to that shape. Double-click to reset.

In [4]:
by_year_shape = (us.groupby(["year", "shape_group"]).size()
                 .reset_index(name="sightings"))
by_year_shape.head()

,year,shape_group,sightings
0,1950,disk,12
1,1950,formation,1
2,1950,light,3
3,1950,other,3
4,1950,oval,4


In [5]:
shape_order = top_shapes + ["other"]
shape_color = alt.Color("shape_group:N", title="Shape",
                        scale=alt.Scale(domain=shape_order, scheme="tableau10"))

brush = alt.selection_interval(encodings=["x"], name="years")
shape_click = alt.selection_point(fields=["shape_group"], name="shape_pick")

timeline = alt.Chart(by_year_shape).mark_area(
    line={"color": "#4c78a8"}, color="#4c78a8", opacity=0.35
).encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
    y=alt.Y("sum(sightings):Q", title="Sightings reported"),
    tooltip=[alt.Tooltip("year:Q", title="Year"),
             alt.Tooltip("sum(sightings):Q", title="Sightings")]
).transform_filter(
    shape_click
).add_params(
    brush
).properties(
    width=600, height=220,
    title="UFO sightings per year (drag to select years; click a shape below to filter)"
)

shape_bars = alt.Chart(by_year_shape).mark_bar().encode(
    x=alt.X("sum(sightings):Q", title="Sightings in selected years"),
    y=alt.Y("shape_group:N", title="Reported shape", sort="-x"),
    color=alt.condition(shape_click, shape_color, alt.value("lightgray")),
    tooltip=[alt.Tooltip("shape_group:N", title="Shape"),
             alt.Tooltip("sum(sightings):Q", title="Sightings")]
).transform_filter(
    brush
).add_params(
    shape_click
).properties(
    width=600, height=220,
    title="Reported shape"
)

plot1 = alt.vconcat(timeline, shape_bars).configure_view(stroke=None)
plot1

alt.VConcatChart(...)

In [6]:
plot1.save("../assets/json/ufo_timeline_shapes.json")

## 4. Plot 2: Where sightings are reported (map)

For the map I round each report's coordinates to the nearest whole degree and count reports per grid cell and shape. That cuts ~65k points down to a few thousand rows. Alaska and Hawaii are kept; the `albersUsa` projection places them in insets.

- **Position:** longitude / latitude (geographic, quantitative)
- **Size and color:** number of sightings in the cell, on a log scale because a few metro areas have far more reports than rural cells.
- **Dropdown:** pick a shape to see if any shape is regional. The aggregation (`sum`) runs after the filter, so the counts update.

In [7]:
us["lat_bin"] = us["latitude"].round(0)
us["lon_bin"] = us["longitude"].round(0)

grid = (us.groupby(["lat_bin", "lon_bin", "shape_group"]).size()
        .reset_index(name="sightings"))
print(len(grid), "rows")
grid.head()

6514 rows


,lat_bin,lon_bin,shape_group,sightings
0,-46.0,170.0,other,1
1,-35.0,-59.0,oval,1
2,-34.0,18.0,disk,1
3,-31.0,22.0,disk,1
4,-29.0,22.0,triangle,1


In [8]:
states = alt.topo_feature("https://cdn.jsdelivr.net/npm/vega-datasets@2.8.0/data/us-10m.json", "states")

shape_dropdown = alt.binding_select(options=["all"] + shape_order, name="Shape: ")
shape_param = alt.param(name="shape_choice", value="all", bind=shape_dropdown)

background = alt.Chart(states).mark_geoshape(
    fill="#f2f2f2", stroke="white", strokeWidth=0.8
)

points = alt.Chart(grid).transform_filter(
    "shape_choice == 'all' || datum.shape_group == shape_choice"
).transform_aggregate(
    sightings="sum(sightings)", groupby=["lat_bin", "lon_bin"]
).mark_circle(opacity=0.75, stroke="white", strokeWidth=0.3).encode(
    longitude="lon_bin:Q",
    latitude="lat_bin:Q",
    size=alt.Size("sightings:Q", title="Sightings",
                  scale=alt.Scale(type="log", range=[4, 220])),
    color=alt.Color("sightings:Q", title="Sightings",
                    scale=alt.Scale(type="log", scheme="orangered")),
    tooltip=[alt.Tooltip("lat_bin:Q", title="Latitude"),
             alt.Tooltip("lon_bin:Q", title="Longitude"),
             alt.Tooltip("sightings:Q", title="Sightings")]
).add_params(shape_param)

plot2 = (background + points).project("albersUsa").properties(
    width=650, height=400,
    title="Where UFO sightings are reported (1° grid, 1950-2013)"
).configure_view(stroke=None)
plot2

alt.LayerChart(...)

In [9]:
plot2.save("../assets/json/ufo_map.json")